# Mumbai Data - Temporal Analysis

Visualize when data was collected.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from pathlib import Path
from datetime import datetime

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
BASE_DIR = Path('../..')

VIDEO_METADATA_FILES = [
    BASE_DIR / '1_8_exif_video_metadata.csv',
    BASE_DIR / '9_11_exif_video_metadata.csv',
    BASE_DIR / '13_19_6_7_exif_video_metadata.csv',
]

## 1. Load Video Metadata

In [ ]:
video_dfs = []
for f in VIDEO_METADATA_FILES:
    if f.exists():
        df = pd.read_csv(f)
        df['source_file'] = f.stem
        video_dfs.append(df)
        print(f"{f.name}: {len(df)} videos")

videos = pd.concat(video_dfs, ignore_index=True)
print(f"\nTotal videos: {len(videos)}")

In [ ]:
videos['recording_datetime'] = pd.to_datetime(
    videos['recording_datetime'], 
    format='%Y:%m:%d %H:%M:%S', 
    errors='coerce'
)

videos['recording_date'] = videos['recording_datetime'].dt.date
videos['recording_hour'] = videos['recording_datetime'].dt.hour
videos['recording_dayofweek'] = videos['recording_datetime'].dt.dayofweek
videos['recording_day_name'] = videos['recording_datetime'].dt.day_name()

videos['end_datetime'] = videos['recording_datetime'] + pd.to_timedelta(videos['video_duration_sec'], unit='s')

print(f"Valid datetime records: {videos['recording_datetime'].notna().sum()}")
print(f"Date range: {videos['recording_datetime'].min()} to {videos['recording_datetime'].max()}")

## 2. Collection Timeline (Gantt Chart)

In [ ]:
videos_sorted = videos.sort_values('recording_datetime').reset_index(drop=True)
videos_sorted = videos_sorted[videos_sorted['recording_datetime'].notna()]

fig, ax = plt.subplots(figsize=(14, max(8, len(videos_sorted) * 0.15)))

source_colors = {
    '1_8_exif_video_metadata': '#1f77b4',
    '9_11_exif_video_metadata': '#ff7f0e',
    '13_19_6_7_exif_video_metadata': '#2ca02c'
}

for idx, row in videos_sorted.iterrows():
    start = row['recording_datetime']
    duration = row['video_duration_sec'] / 3600
    color = source_colors.get(row['source_file'], 'gray')
    
    ax.barh(idx, duration, left=mdates.date2num(start), height=0.8, color=color, alpha=0.7)

ax.set_yticks(range(len(videos_sorted)))
ax.set_yticklabels(videos_sorted['video_id'], fontsize=6)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
ax.xaxis.set_major_locator(mdates.DayLocator(interval=7))
plt.xticks(rotation=45, ha='right')

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, label=k.replace('_exif_video_metadata', '')) 
                   for k, c in source_colors.items()]
ax.legend(handles=legend_elements, loc='upper right', title='Source Folder')

ax.set_xlabel('Date')
ax.set_ylabel('Video ID')
ax.set_title('Video Collection Timeline')

plt.tight_layout()
plt.show()

## 3. Videos Per Day

In [ ]:
daily_counts = videos.groupby('recording_date').agg({
    'video_id': 'count',
    'video_duration_sec': 'sum'
}).rename(columns={'video_id': 'num_videos', 'video_duration_sec': 'total_duration_sec'})
daily_counts['total_duration_min'] = daily_counts['total_duration_sec'] / 60

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

axes[0].bar(daily_counts.index, daily_counts['num_videos'], color='steelblue', alpha=0.7)
axes[0].set_ylabel('Number of Videos')
axes[0].set_title('Videos Recorded Per Day')

axes[1].bar(daily_counts.index, daily_counts['total_duration_min'], color='coral', alpha=0.7)
axes[1].set_ylabel('Total Duration (minutes)')
axes[1].set_xlabel('Date')
axes[1].set_title('Total Recording Duration Per Day')

plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print(f"\nDaily Statistics:")
print(daily_counts.sort_index())

## 4. Time of Day Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

hourly_counts = videos['recording_hour'].value_counts().sort_index()
all_hours = pd.Series(0, index=range(24))
all_hours.update(hourly_counts)

colors = ['#2c3e50' if 6 <= h < 18 else '#34495e' for h in range(24)]
axes[0].bar(all_hours.index, all_hours.values, color=colors, alpha=0.8)
axes[0].set_xlabel('Hour of Day')
axes[0].set_ylabel('Number of Videos')
axes[0].set_title('Videos by Hour of Day')
axes[0].set_xticks(range(0, 24, 2))
axes[0].axvspan(6, 18, alpha=0.1, color='yellow', label='Daytime (6AM-6PM)')
axes[0].legend()

hourly_duration = videos.groupby('recording_hour')['video_duration_sec'].sum() / 60
all_hours_dur = pd.Series(0.0, index=range(24))
all_hours_dur.update(hourly_duration)

axes[1].bar(all_hours_dur.index, all_hours_dur.values, color='coral', alpha=0.8)
axes[1].set_xlabel('Hour of Day')
axes[1].set_ylabel('Total Duration (minutes)')
axes[1].set_title('Recording Duration by Hour of Day')
axes[1].set_xticks(range(0, 24, 2))
axes[1].axvspan(6, 18, alpha=0.1, color='yellow')

plt.tight_layout()
plt.show()

print(f"\nPeak recording hours:")
print(hourly_counts.nlargest(5))

## 5. Day of Week Distribution

In [ ]:
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
dow_counts = videos['recording_day_name'].value_counts().reindex(day_order).fillna(0)

fig, ax = plt.subplots(figsize=(10, 5))

colors = ['#e74c3c' if d in ['Saturday', 'Sunday'] else '#3498db' for d in day_order]
bars = ax.bar(dow_counts.index, dow_counts.values, color=colors, alpha=0.8)

ax.set_xlabel('Day of Week')
ax.set_ylabel('Number of Videos')
ax.set_title('Videos by Day of Week')

for bar, count in zip(bars, dow_counts.values):
    if count > 0:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, 
                f'{int(count)}', ha='center', va='bottom')

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#3498db', label='Weekday'),
    Patch(facecolor='#e74c3c', label='Weekend')
]
ax.legend(handles=legend_elements)

plt.tight_layout()
plt.show()

## 6. Collection by Source Folder

In [ ]:
source_stats = videos.groupby('source_folder').agg({
    'video_id': 'count',
    'video_duration_sec': ['sum', 'mean'],
    'recording_datetime': ['min', 'max']
}).round(1)

source_stats.columns = ['num_videos', 'total_duration_sec', 'avg_duration_sec', 'first_recording', 'last_recording']
source_stats['total_duration_min'] = (source_stats['total_duration_sec'] / 60).round(1)
source_stats['avg_duration_min'] = (source_stats['avg_duration_sec'] / 60).round(1)

print("Collection Statistics by Source Folder:")
print(source_stats[['num_videos', 'total_duration_min', 'avg_duration_min', 'first_recording', 'last_recording']])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

source_stats_sorted = source_stats.sort_values('num_videos', ascending=True)

axes[0].barh(source_stats_sorted.index.astype(str), source_stats_sorted['num_videos'], color='steelblue', alpha=0.8)
axes[0].set_xlabel('Number of Videos')
axes[0].set_ylabel('Source Folder')
axes[0].set_title('Videos by Source Folder')

axes[1].barh(source_stats_sorted.index.astype(str), source_stats_sorted['total_duration_min'], color='coral', alpha=0.8)
axes[1].set_xlabel('Total Duration (minutes)')
axes[1].set_ylabel('Source Folder')
axes[1].set_title('Recording Duration by Source Folder')

plt.tight_layout()
plt.show()

## 7. Video Duration Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

duration_min = videos['video_duration_sec'] / 60

axes[0].hist(duration_min.dropna(), bins=30, color='steelblue', alpha=0.7, edgecolor='black')
axes[0].axvline(duration_min.mean(), color='red', linestyle='--', label=f'Mean: {duration_min.mean():.1f} min')
axes[0].axvline(duration_min.median(), color='green', linestyle=':', label=f'Median: {duration_min.median():.1f} min')
axes[0].set_xlabel('Duration (minutes)')
axes[0].set_ylabel('Number of Videos')
axes[0].set_title('Video Duration Distribution')
axes[0].legend()

axes[1].boxplot([duration_min.dropna()], vert=True)
axes[1].set_ylabel('Duration (minutes)')
axes[1].set_title('Video Duration Box Plot')
axes[1].set_xticklabels(['All Videos'])

plt.tight_layout()
plt.show()

print(f"\nDuration Statistics:")
print(f"  Min: {duration_min.min():.1f} min")
print(f"  Max: {duration_min.max():.1f} min")
print(f"  Mean: {duration_min.mean():.1f} min")
print(f"  Median: {duration_min.median():.1f} min")
print(f"  Total: {duration_min.sum():.1f} min ({duration_min.sum()/60:.1f} hours)")

## 8. Summary

In [ ]:
print("=" * 50)
print("TEMPORAL ANALYSIS SUMMARY")
print("=" * 50)

print(f"\nCollection Period:")
print(f"  First recording: {videos['recording_datetime'].min()}")
print(f"  Last recording: {videos['recording_datetime'].max()}")
num_days = (videos['recording_datetime'].max() - videos['recording_datetime'].min()).days + 1
print(f"  Span: {num_days} days")

print(f"\nCollection Volume:")
print(f"  Total videos: {len(videos)}")
print(f"  Total duration: {videos['video_duration_sec'].sum()/3600:.1f} hours")
print(f"  Unique days: {videos['recording_date'].nunique()}")

print(f"\nPeak Times:")
peak_hour = videos['recording_hour'].mode().iloc[0] if len(videos['recording_hour'].mode()) > 0 else 'N/A'
peak_day = videos['recording_day_name'].mode().iloc[0] if len(videos['recording_day_name'].mode()) > 0 else 'N/A'
print(f"  Most common hour: {peak_hour}:00")
print(f"  Most common day: {peak_day}")